In [ ]:

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import pad_sequences

# 1. Load and preprocess data
df = pd.read_csv("LP IV Updated Datasets/CBOW/IMDB Dataset.csv").dropna()

df["words"] = df.review.apply(preprocess_string)
y = df.sentiment.map({"negative": 0, "positive": 1})

train, test, y_train, y_test = train_test_split(
    df.words.tolist(), y, test_size=0.2
)

# 2. Train CBOW and create embeddings
cbow = Word2Vec(train, min_count=1, sg=0, epochs=20)

train = pad_sequences(train, maxlen=300, dtype=object, value="", padding="post")
test = pad_sequences(test, maxlen=300, dtype=object, value="", padding="post")

x_train = np.array([cbow.wv.get_mean_vector(w) for w in train])
x_test = np.array([cbow.wv.get_mean_vector(w) for w in test])

# 3. Create neural network
model = Sequential([
    Input(shape=(100,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(x_train, y_train, epochs=20, batch_size=128)

# 4. Evaluate
loss, accuracy = model.evaluate(x_test, y_test)
print("Accuracy:", accuracy * 100, "%")
